# Profundidad y ResNet con PyTorch

Compararemos CNN sencillas y profundas en CIFAR-10. Después construiremos un bloque residual y probaremos ResNet50 con su clasificador adaptado a diez clases.

## Entorno y datos

Ejecuta con el kernel **IPD434**. PyTorch utilizará CUDA si está disponible. La primera carga de CIFAR-10 descarga el conjunto a `../data/cifar10`. Usamos subconjuntos para que la comparación sea rápida.

In [ ]:
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report
from torch import nn
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, models, transforms

SEED = 2026
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| dispositivo:', device)

In [ ]:
DATA_ROOT = Path('../data/cifar10')
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
train_full = datasets.CIFAR10(DATA_ROOT, train=True, download=True, transform=transform)
test_full = datasets.CIFAR10(DATA_ROOT, train=False, download=True, transform=transform)
generator = torch.Generator().manual_seed(SEED)
train_set, val_set, unused = random_split(train_full, [5000, 1000, 44000], generator=generator)
test_indices = torch.randperm(len(test_full), generator=torch.Generator().manual_seed(SEED + 1))[:2000].tolist()
test_set = Subset(test_full, test_indices)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True, generator=torch.Generator().manual_seed(SEED), num_workers=0, pin_memory=device.type == 'cuda')
train_eval_loader = DataLoader(train_set, batch_size=256, shuffle=False, num_workers=0)
val_loader = DataLoader(val_set, batch_size=256, shuffle=False, num_workers=0)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=0)
class_names = ['avión', 'automóvil', 'ave', 'gato', 'ciervo', 'perro', 'rana', 'caballo', 'barco', 'camión']
print('Particiones:', len(train_set), len(val_set), len(test_set))

## CNN plana: comparar profundidad

Las tres redes apilan convoluciones antes del clasificador. Comparamos una CNN sencilla, una más profunda y una red sobrecompleja. La validación controla la selección de época.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(3, 32, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(32 * 15 * 15, 32), nn.ReLU(),
            nn.Linear(32, 10),
        )
    def forward(self, x): return self.layers(x)

class DeepCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3), nn.ReLU(),
            nn.Conv2d(32, 32, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3), nn.ReLU(),
            nn.Conv2d(64, 64, 3), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(64 * 5 * 5, 64), nn.ReLU(), nn.Linear(64, 10))
    def forward(self, x): return self.classifier(self.features(x))

class OverdeepCNN(nn.Module):
    def __init__(self, extra_layers=10):
        super().__init__()
        layers = [nn.Conv2d(3, 32, 3), nn.ReLU()]
        for _ in range(extra_layers):
            layers += [nn.Conv2d(32, 32, 3, padding=1), nn.ReLU()]
        layers += [nn.MaxPool2d(2), nn.Flatten(), nn.Linear(32 * 15 * 15, 64), nn.ReLU(), nn.Linear(64, 10)]
        self.layers = nn.Sequential(*layers)
    def forward(self, x): return self.layers(x)

criterion = nn.CrossEntropyLoss()
MAX_EPOCHS, PATIENCE = 5, 2

def evaluate(model, loader):
    model.eval(); loss_sum = 0.0; correct = 0; total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images); loss = criterion(logits, labels)
            loss_sum += loss.item() * len(labels)
            correct += (logits.argmax(1) == labels).sum().item(); total += len(labels)
    return loss_sum / total, correct / total

def fit_model(model, epochs=MAX_EPOCHS):
    model = model.to(device); optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    best_loss = float('inf'); best_state = deepcopy(model.state_dict()); waiting = 0
    history = {'train_loss': [], 'val_loss': [], 'val_accuracy': []}
    for epoch in range(1, epochs + 1):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(images), labels); loss.backward(); optimizer.step()
        train_loss, _ = evaluate(model, train_eval_loader)
        val_loss, val_accuracy = evaluate(model, val_loader)
        history['train_loss'].append(train_loss); history['val_loss'].append(val_loss); history['val_accuracy'].append(val_accuracy)
        if val_loss < best_loss - 1e-4:
            best_loss = val_loss; best_state = deepcopy(model.state_dict()); waiting = 0
        else:
            waiting += 1
            if waiting >= PATIENCE: break
    model.load_state_dict(best_state)
    return model, history

In [ ]:
plain_models = {
    'sencilla': SimpleCNN(),
    'profunda': DeepCNN(),
    'muy profunda': OverdeepCNN(extra_layers=10),
}
plain_histories = {}
for name, candidate in plain_models.items():
    print('Entrenando CNN', name)
    plain_models[name], plain_histories[name] = fit_model(candidate)
    print('Mejor exactitud de validación:', max(plain_histories[name]['val_accuracy']))

## Bloque residual y ResNet pequeña

El atajo suma la entrada del bloque a la transformación convolucional. Si cambia el número de canales o la resolución, una proyección $1\times1$ adapta el atajo.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.projection = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.projection = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        shortcut = self.projection(x)
        y = torch.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        return torch.relu(y + shortcut)

class MiniResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU())
        self.blocks = nn.Sequential(
            ResidualBlock(32, 32), ResidualBlock(32, 32),
            ResidualBlock(32, 64, stride=2), ResidualBlock(64, 64),
            ResidualBlock(64, 128, stride=2), ResidualBlock(128, 128),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.stem(x); x = self.blocks(x); x = self.pool(x).flatten(1)
        return self.classifier(x)

resnet = MiniResNet().to(device)
print('Salida ResNet:', resnet(torch.zeros(2, 3, 32, 32, device=device)).shape)

In [ ]:
resnet, resnet_history = fit_model(MiniResNet())
print('Mejor exactitud de validación ResNet:', max(resnet_history['val_accuracy']))

## ResNet50 desde cero

Torchvision ofrece ResNet50. Reemplazamos la capa final para las diez clases de CIFAR-10. Este bloque comprueba un paso de entrenamiento; entrenar desde cero requiere más épocas y cómputo.

In [ ]:
resnet50 = models.resnet50(weights=None)
resnet50.fc = nn.Linear(resnet50.fc.in_features, 10)
resnet50 = resnet50.to(device)
resnet_optimizer = torch.optim.Adam(resnet50.parameters(), lr=1e-4)
images, labels = next(iter(DataLoader(train_set, batch_size=4, shuffle=False)))
images, labels = images.to(device), labels.to(device)
resnet50.train()
resnet_optimizer.zero_grad(set_to_none=True)
loss = criterion(resnet50(images), labels)
loss.backward()
resnet_optimizer.step()
print('Paso ResNet50 | cross entropy:', loss.item())